## Project 6: Sports Image Classification

For the last project we build one neural-network next we're going to build one CNN for this classification

In [2]:
#importing the packages
import torch 
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import os

In [3]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [3]:
#preparing the dataset
data_dir = "100sportsdataset"
train_dir = datasets.ImageFolder(root=os.path.join(data_dir, "train"), transform=transform)
val_dir = datasets.ImageFolder(root=os.path.join(data_dir, "valid"), transform=transform)
test_dir = datasets.ImageFolder(root=os.path.join(data_dir, "test"), transform=transform)

In [4]:
train_loader = DataLoader(train_dir, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dir, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dir, batch_size=16, shuffle=True)

Next we will create our own CNN model to train, the model architecture for CNN will look like below:

32 -> 64 -> 128 -> 256 -> 512 

and the fc layers should look like below

128 -> 512 -> 256 -> 128 -> num_classes 

In [5]:
#defining the cnn architecture
class SportsCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn_layer = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.fc_layer = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512*5*5, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.cnn_layer(x)
        # print("After CNN:",x.shape) # for debugging
        x = self.fc_layer(x)
        return x

In [6]:
model = SportsCNN(num_classes=len(train_dir.classes))

In [7]:
criterion = nn.CrossEntropyLoss()

In [8]:
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [9]:
# scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

Training and Validating the model

In [10]:
for epoch in range(100):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        outputs = model(images)
        loss = criterion(outputs, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    #validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
        # scheduler.step()
    print(f"Epoch {epoch+1}, Train Loss: {running_loss/len(train_loader):.4f}, Val Loss: {val_loss/len(val_loader):.4f}, Val Accuracy: {100 * correct / total:.2f}%")

Epoch 1, Train Loss: 4.1578, Val Loss: 3.5989, Val Accuracy: 14.80%
Epoch 2, Train Loss: 3.5977, Val Loss: 3.0770, Val Accuracy: 23.80%
Epoch 3, Train Loss: 3.2842, Val Loss: 2.8103, Val Accuracy: 32.20%
Epoch 4, Train Loss: 3.0747, Val Loss: 2.5750, Val Accuracy: 35.40%
Epoch 5, Train Loss: 2.8896, Val Loss: 2.3004, Val Accuracy: 42.20%
Epoch 6, Train Loss: 2.7596, Val Loss: 2.1862, Val Accuracy: 43.40%
Epoch 7, Train Loss: 2.6237, Val Loss: 2.1187, Val Accuracy: 46.60%
Epoch 8, Train Loss: 2.4722, Val Loss: 2.0315, Val Accuracy: 48.40%
Epoch 9, Train Loss: 2.3786, Val Loss: 1.7974, Val Accuracy: 52.20%
Epoch 10, Train Loss: 2.2673, Val Loss: 1.7060, Val Accuracy: 51.60%
Epoch 11, Train Loss: 2.1712, Val Loss: 1.6245, Val Accuracy: 55.00%
Epoch 12, Train Loss: 2.0837, Val Loss: 1.7436, Val Accuracy: 54.60%
Epoch 13, Train Loss: 1.9829, Val Loss: 1.4565, Val Accuracy: 60.80%
Epoch 14, Train Loss: 1.9322, Val Loss: 1.4273, Val Accuracy: 61.40%
Epoch 15, Train Loss: 1.8596, Val Loss: 1.4

In [11]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

test_accuracy = 100 * correct / total
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 80.40%


In [12]:
torch.save(model.state_dict(), "sports_cnn.pth")
print("Model saved!!")

Model saved!!
